In [1]:
!pip install confluent_kafka
!pip install pyspark==3.5.2
!pip install elasticsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 37.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 5.1 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.9 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.2-py2.py3-none-any.whl size=317812369 sha256=2fef9d0670ef36cc17d12266b2220a4cad1c32f3447b8a7b35504fcf0b139f00
  Stored in directory: /root/.cache/pip/wheels/cf/c0/b9/f147f4220fd1d9277d0981b88b35b26f03ad910fffd60013a6
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are in

In [32]:
# ============================================================
# TODO在这里: Sentiment Analysis 可以用多个模型来对比
# ============================================================

def analyze_sentiment(text):
    if not text:
        return "neutral"
    # TODO: Replace with actual sentiment analysis
    return "neutral"

sentiment_udf = udf(analyze_sentiment, StringType())


In [37]:
# # ============================================================
# # 清空 Checkpoint, 从这个目录是用来记录读到哪一条了
# # ============================================================
# import shutil

# # 删除整个 checkpoint 目录
# checkpoint_dir = "/kaggle/working/checkpoints"
# shutil.rmtree(checkpoint_dir, ignore_errors=True)

# # 也删掉旧的 checkpoint.txt（如果存在）
# import os
# if os.path.exists("/kaggle/working/checkpoint.txt"):
#     os.remove("/kaggle/working/checkpoint.txt")

In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from elasticsearch import Elasticsearch, helpers
import json
import os

# ============================================================
# Configuration
# ============================================================

config = {
    "kafka": {
        "bootstrap.servers": "pkc-ox31np.ap-southeast-7.aws.confluent.cloud:9092",
        "security.protocol": "SASL_SSL",
        "sasl.mechanisms": "PLAIN",
        "sasl.username": "S4AU73FQKMOAMSPL",
        "sasl.password": "cfltEuKzwF9VDz4Y2XwilhMpyc+9rxUP4xYfpOFYbbC1VefzzNc8rOYHfIavyVLw",
    },
    "elasticsearch": {
        "host": "https://20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com:443",
        "api_key": "eEFjNGJaMEJfQXBJNldXZFc3YU06RWhYdmpGZ1lsNzZlYWJJa1QwMXdQdw==",
        "index": "enron_emails"
    }
}

# Checkpoint directory
checkpoint_dir = "/kaggle/working/checkpoints/enron_kafka_es"
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

# ============================================================
# Schema for Enron emails (matches your Kafka data)
# ============================================================

email_schema = StructType([
    StructField("file_path", StringType(), True),
    StructField("message_id", StringType(), True),
    StructField("sent_at", StringType(), True),
    StructField("sender", StringType(), True),
    StructField("to", StringType(), True),
    StructField("cc", StringType(), True),
    StructField("bcc", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("x_folder", StringType(), True),
    StructField("x_origin", StringType(), True),
    StructField("x_filename", StringType(), True),
    StructField("body", StringType(), True),
    StructField("body_length", IntegerType(), True)
])

In [40]:
# ============================================================
# 清空已有的 Elasticsearch 索引
# ============================================================
es_client = Elasticsearch(
    config['elasticsearch']['host'],
    api_key=config['elasticsearch']['api_key'],
    verify_certs=False
)

index_name = "enron_emails"

if es_client.indices.exists(index=index_name):
    # 删除索引
    es_client.indices.delete(index=index_name)
    print(f"Deleted index: {index_name}")
else:
    print(f"Index {index_name} does not exist")


Deleted index: enron_emails


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [41]:

# ============================================================
# Elasticsearch client (for manual bulk writes)
# ============================================================
es_client = Elasticsearch(
    config['elasticsearch']['host'],
    api_key=config['elasticsearch']['api_key'],
    verify_certs=False
)

index_name = "enron_emails"

# create index mapping
mapping = {
    "mappings": {
        "properties": {
            "bcc": {"type": "keyword"},
            "body": {"type": "text"},
            "body_length": {"type": "integer"},
            "cc": {"type": "keyword"},
            "file_path": {"type": "keyword"},
            "message_id": {"type": "keyword"},
            "sender": {"type": "keyword"},
            "sent_at": {
                "type": "date",
                "format": "yyyy-MM-dd HH:mm:ss||yyyy-MM-dd'T'HH:mm:ss.SSSZ||strict_date_optional_time||epoch_millis"
            },
            "sentiment": {"type": "keyword"},
            "subject": {
                "type": "text",
                "fields": {
                    "keyword": {
                        "type": "keyword",
                        "ignore_above": 256
                    }
                }
            },
            "to": {"type": "keyword"},
            "x_filename": {"type": "keyword"},
            "x_folder": {"type": "keyword"},
            "x_origin": {"type": "keyword"}
        }
    }
}

es_client.indices.create(index=index_name, body=mapping)

# # test
# test_doc = {
#     "sent_at": "2001-05-14 23:39:00",
#     "message_id": "test_123",
#     "sender": "test@enron.com",
#     "to": "test@enron.com",
#     "subject": "Test",
#     "body": "Test message",
#     "sentiment": "neutral"
# }

# result = es_client.index(index=index_name, document=test_doc, refresh=True)
# print(f"Test passed! Document ID: {result['_id']}")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'enron_emails'})

In [42]:
def write_to_es(df, epoch_id):
    if df.count() == 0:
        return
    
    # 转换 sent_at 日期格式
    if 'sent_at' in df.columns:
        from pyspark.sql.functions import regexp_replace, to_timestamp, date_format
        df = df.withColumn('sent_at', 
            regexp_replace(col('sent_at'), ' UTC$', ''))
        df = df.withColumn('sent_at', 
            to_timestamp(col('sent_at'), 'yyyy-MM-dd HH:mm:ss'))
        df = df.withColumn('sent_at',
            date_format(col('sent_at'), "yyyy-MM-dd'T'HH:mm:ss.SSSZ"))
    
    records = df.toJSON().collect()
    actions = []
    for record in records:
        actions.append({
            "_index": config['elasticsearch']['index'],
            "_source": json.loads(record)
        })
    
    if actions:
        print(f"Sample record: {actions[0]['_source']}")
        
        success, failed = helpers.bulk(es_client, actions, stats_only=False, raise_on_error=False)
        print(f"Batch {epoch_id}: {success} succeeded, {len(failed)} failed")
        
        if failed:
            print(f"First failed record: {actions[0]['_source']}")
            print(f"Error details: {failed[0]}")

In [43]:
# ============================================================
# Create Spark Session
# ============================================================

spark = SparkSession.builder \
    .appName("EnronKafkaToES") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.2") \
    .getOrCreate()

# 设置日志级别为 ERROR 或 FATAL 来屏蔽 WARN
spark.sparkContext.setLogLevel("ERROR")

# ============================================================
# Read from Kafka and write to Elasticsearch
# ============================================================

def read_from_kafka_and_write_to_es(spark):
    topic = "raw_emails_topic"
    
    stream_df = (spark.readStream
                 .format("kafka")
                 .option("kafka.bootstrap.servers", config['kafka']['bootstrap.servers'])
                 .option("subscribe", topic)
                 .option("startingOffsets", "earliest")
                 .option("kafka.security.protocol", config['kafka']['security.protocol'])
                 .option("kafka.sasl.mechanism", config['kafka']['sasl.mechanisms'])
                 .option("kafka.sasl.jaas.config",
                        f'org.apache.kafka.common.security.plain.PlainLoginModule required username="{config["kafka"]["sasl.username"]}" '
                        f'password="{config["kafka"]["sasl.password"]}";')
                 .option("failOnDataLoss", "false")
                 .option("maxOffsetsPerTrigger", 100)
                 .load()
                )
    
    # Parse JSON
    parsed_df = stream_df.select(
        from_json(col('value').cast("string"), email_schema).alias("data")
    ).select("data.*")
    
    # Add sentiment (placeholder)
    enriched_df = parsed_df.withColumn("sentiment", sentiment_udf(col("body")))
    
    # Write to Elasticsearch via foreachBatch
    query = (enriched_df.writeStream
             .foreachBatch(write_to_es)
             .outputMode("append")
             .trigger(processingTime="10 seconds")
             .option("checkpointLocation", checkpoint_dir)
             .start()
             .awaitTermination()
            )

In [44]:
# 一直运行就会一直流处理    
if __name__ == "__main__":
    read_from_kafka_and_write_to_es(spark)

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Sample record: {'file_path': 'allen-p/_sent_mail/1.', 'message_id': '<18782981.1075855378110.JavaMail.evans@thyme>', 'sent_at': '2001-05-14T23:39:00.000+0000', 'sender': 'phillip.allen@enron.com', 'to': 'tim.belden@enron.com', 'cc': '', 'bcc': '', 'subject': '', 'x_folder': "\\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\'Sent Mail", 'x_origin': 'Allen-P', 'x_filename': 'pallen (Non-Privileged).pst', 'body': 'Here is our forecast', 'body_length': 20, 'sentiment': 'neutral'}
Batch 0: 96 succeeded, 0 failed
Sample record: {'file_path': 'allen-p/_sent_mail/200.', 'message_id': '<27259145.1075855689639.JavaMail.evans@thyme>', 'sent_at': '2000-08-07T07:06:00.000+0000', 'sender': 'phillip.allen@enron.com', 'to': 'mike.grigsby@enron.com, keith.holst@enron.com, frank.ermis@enron.com, jane.tholt@enron.com, steven.south@enron.com', 'cc': '', 'bcc': '', 'subject': 'Gas fundamentals development website', 'x_folder': "\\Phillip_Allen_Dec2000\\Notes Folders\\'sent mail", 'x_origin': 'Allen-P', 'x_fil

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 1: 96 succeeded, 0 failed
Sample record: {'file_path': 'allen-p/_sent_mail/29.', 'message_id': '<21295362.1075855379019.JavaMail.evans@thyme>', 'sent_at': '2001-04-25T00:26:00.000+0000', 'sender': 'phillip.allen@enron.com', 'to': 'eric.benson@enron.com', 'cc': '', 'bcc': '', 'subject': 'Re: Instructions for FERC Meetings', 'x_folder': "\\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\'Sent Mail", 'x_origin': 'Allen-P', 'x_filename': 'pallen (Non-Privileged).pst', 'body': 'it works.  thank you', 'body_length': 20, 'sentiment': 'neutral'}


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

Sample record: {'file_path': 'allen-p/_sent_mail/419.', 'message_id': '<28589107.1075855725265.JavaMail.evans@thyme>', 'sent_at': '2001-03-29T09:41:00.000+0000', 'sender': 'phillip.allen@enron.com', 'to': 'scsatkfa@caprock.net', 'cc': '', 'bcc': '', 'subject': '', 'x_folder': "\\Phillip_Allen_June2001\\Notes Folders\\'sent mail", 'x_origin': 'Allen-P', 'x_filename': 'pallen.nsf', 'body': 'Cary,\n\nHere is the picture of the house I have in mind.  I was going for a simple \nfarmhouse style to  place on 5 acres near Wimberley.  \n\nA few points that might not be obvious from the plans are:\n\nThere will be a double porch across the back just like the front\nNo dormers (Metal roof)\n1/2 bath under stairs\nOverall dimensions are 55 by 40\n\n\nWhat I am looking for is a little design help in the kitchen.  More cabinets, \nmaybe a different shaped island, and a way to enlarge the pantry.    Reagan \nsuggested that I find a way to make the exterior more attractive.  I want to \nkeep a simple 

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 3: 96 succeeded, 0 failed


In [45]:
# 查询最新的5条记录
response = es_client.search(
    index="enron_emails",
    body={
        "query": {"match_all": {}},
        "sort": [{"sent_at": {"order": "desc"}}],
        "size": 5
    }
)

# 打印结果
print(f"Total records: {response['hits']['total']['value']}")
print("\n=== Latest 5 Records ===\n")
for i, hit in enumerate(response['hits']['hits'], 1):
    source = hit['_source']
    print(f"{i}. ID: {hit['_id']}")
    print(f"   Sent at: {source.get('sent_at')}")
    print(f"   From: {source.get('sender')}")
    print(f"   To: {source.get('to')}")
    print(f"   Subject: {source.get('subject', '')[:50]}")
    print(f"   Sentiment: {source.get('sentiment')}")
    print("-" * 50)

Total records: 384

=== Latest 5 Records ===

1. ID: qAeAbZ0B_ApI6WWdi9mV
   Sent at: 2001-05-14T23:39:00.000+0000
   From: phillip.allen@enron.com
   To: tim.belden@enron.com
   Subject: 
   Sentiment: neutral
--------------------------------------------------
2. ID: OzWAbZ0BpeIanCvLkCwe
   Sent at: 2001-05-14T20:39:00.000+0000
   From: phillip.allen@enron.com
   To: outlook.team@enron.com
   Subject: Re: 2- SURVEY/INFORMATION EMAIL 5-14- 01
   Sentiment: neutral
--------------------------------------------------
3. ID: 9zWAbZ0BpeIanCvLzSy0
   Sent at: 2001-05-14T13:39:00.000+0000
   From: phillip.allen@enron.com
   To: tim.belden@enron.com
   Subject: 
   Sentiment: neutral
--------------------------------------------------
4. ID: izWAbZ0BpeIanCvLpiyk
   Sent at: 2001-05-14T08:39:00.000+0000
   From: phillip.allen@enron.com
   To: outlook.team@enron.com
   Subject: Re: 2- SURVEY/INFORMATION EMAIL 5-14- 01
   Sentiment: neutral
--------------------------------------------------
5. ID:

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Sample record: {'file_path': 'allen-p/_sent_mail/521.', 'message_id': '<8058476.1075855727502.JavaMail.evans@thyme>', 'sent_at': '2001-02-06T14:12:00.000+0000', 'sender': 'phillip.allen@enron.com', 'to': 'mike.grigsby@enron.com', 'cc': '', 'bcc': '', 'subject': 'Smeltering', 'x_folder': "\\Phillip_Allen_June2001\\Notes Folders\\'sent mail", 'x_origin': 'Allen-P', 'x_filename': 'pallen.nsf', 'body': '---------------------- Forwarded by Phillip K Allen/HOU/ECT on 02/06/2001 \n02:12 PM ---------------------------\n   \n\tEnron North America Corp.\n\t\n\tFrom:  Frank Hayden @ ENRON                           02/06/2001 12:06 PM\n\t\n\nTo: Tim Belden/HOU/ECT@ECT, Chris Gaskill/Corp/Enron@Enron, Phillip K \nAllen/HOU/ECT@ECT, John J Lavorato/Corp/Enron, Kevin M Presto/HOU/ECT@ECT\ncc: LaCrecia Davenport/Corp/Enron@Enron, Bharat Khanna/NA/Enron@Enron \nSubject: Smeltering\n\n\nBelow are some articles relating to aluminum, power and gas prices.  Thought \nit would be of interest.\nFrank\n\n---\

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 4: 96 succeeded, 0 failed
Sample record: {'file_path': 'allen-p/_sent_mail/587.', 'message_id': '<2727013.1075855729082.JavaMail.evans@thyme>', 'sent_at': '2000-12-27T16:00:00.000+0000', 'sender': 'phillip.allen@enron.com', 'to': 'jim123@pdq.net', 'cc': '', 'bcc': '', 'subject': '', 'x_folder': "\\Phillip_Allen_June2001\\Notes Folders\\'sent mail", 'x_origin': 'Allen-P', 'x_filename': 'pallen.nsf', 'body': 'Jim,\n\n I would appreciate your help in locating financing for the project I \ndescribed to you last week.  The project is a 134 unit apartment complex in \nSan Marcos.  There will be a builder/developer plus myself and possibly a \ncouple of other investors involved.  As I mentioned last week, I would like \nto find interim financing (land, construction, semi-perm) that does not \nrequire the investors to personally guarantee.  If there is a creative way to \nstructure the deal, I would like to hear your suggestions.  One idea that has \nbeen mentioned is to obtain a "forwar

...

Sample record: {'file_path': 'arora-h/deleted_items/129.', 'message_id': '<25525631.1075861395057.JavaMail.evans@thyme>', 'sent_at': '2001-11-23T09:10:55.000+0000', 'sender': 'arsystem@mailman.enron.com', 'to': 'harry.arora@enron.com', 'cc': '', 'bcc': '', 'subject': 'Your Approval is Overdue: Access Request for jaime.gualy@enron.com', 'x_folder': '\\HARORA (Non-Privileged)\\Arora, Harry\\Deleted Items', 'x_origin': 'Arora-H', 'x_filename': 'HARORA (Non-Privileged).pst', 'body': 'This request has been pending your approval for  114 days.  Please click http://itcapps.corp.enron.com/srrs/auth/emailLink.asp?ID=000000000041547&Page=Approval to review and act upon this request.\n\n\n\n\n\nRequest ID          : 000000000041547\nRequest Create Date : 6/19/01 7:50:30 AM\nRequested For       : jaime.gualy@enron.com\nResource Name       : \\\\nahoutrd\\houston\\pwr\\common\\Electric - [Read]\nResource Type       : Directory', 'body_length': 426, 'sentiment': 'neutral'}


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 83: 96 succeeded, 0 failed


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Sample record: {'file_path': 'arora-h/deleted_items/192.', 'message_id': '<22853673.1075861397924.JavaMail.evans@thyme>', 'sent_at': '2001-11-27T17:55:03.000+0000', 'sender': 'feedback@intcx.com', 'to': 'powerindex@list.intcx.com', 'cc': '', 'bcc': '', 'subject': 'Power Indices', 'x_folder': '\\HARORA (Non-Privileged)\\Arora, Harry\\Deleted Items', 'x_origin': 'Arora-H', 'x_filename': 'HARORA (Non-Privileged).pst', 'body': '=\n                                                                           =\n                                                                           =\n                                                                           =\n                                                                           =\n                                                                           =\n                                                                           =\n                                                                           =\n                       